# 04 · Model semantic spaces

Narrative only: the logic lives in `src/embeddings/`. Run `python run.py embeddings` first. Conditions whose models could not be run are listed as **PENDING USER RUN** with the command that produces them.

In [ ]:
import json, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd
import config
from src.utils.concept_order import load_concept_order
from src.utils.matrices import load_matrix
load = lambda name: json.loads((config.RESULTS_DIR / name).read_text())

## 1. Which conditions exist (coverage is a result, D-15)

In [ ]:
pd.read_csv(config.RESULTS_DIR / 'embedding_coverage.csv')

## 2. word2vec lookup (D-15, D-46)
Every gloss's lookup rule is recorded; Google News is American English and dropped common function words.

In [ ]:
t = pd.read_csv(config.RESULTS_DIR / 'embedding_lookup_w2v_gnews.csv')
print(t['rule'].value_counts().to_dict())
t[t['rule'] != 'exact']

## 3. The multilingual input (D-17, D-48)
Forms are CLICS⁴ orthography; transcription datasets are rejected automatically.

In [ ]:
cov = load('translated_coverage.json')
{k: v for k, v in cov.items() if k != 'dataset_choice'}

In [ ]:
pd.DataFrame(cov['dataset_choice'])

In [ ]:
f = pd.read_csv(config.RESULTS_DIR / 'translated_forms.csv')
f[f['concept_id'].isin(['YESTERDAY','TOMORROW','HAND','ARM','TREE','WOOD','TONGUE','LANGUAGE'])].pivot(index='concept_id', columns='iso', values='form_used')

Hindi कल appears for both YESTERDAY and TOMORROW: the *kal* colexification reaches the multilingual-input condition as two identical input strings.

## 4. Similarity matrices (D-47)
Centring removes the common direction that inflates every cosine.

In [ ]:
pd.read_csv(config.RESULTS_DIR / 'model_similarity_summary.csv')

In [ ]:
order = load_concept_order()
idx = {c: i for i, c in enumerate(order)}
m = load_matrix(config.MATRIX_DIR / 'model_w2v_gnews.npz', order).values
pairs = [('ARM','HAND'),('TREE','WOOD'),('TOMORROW','YESTERDAY'),('FINGER','TOE'),('SEE','UNDERSTAND'),('LANGUAGE','TONGUE')]
pd.Series({f'{a}~{b}': m[idx[a], idx[b]] for a, b in pairs}, name='word2vec centred cosine').round(3)

These six numbers are descriptive only. Whether English-present pairs are closer than matched English-absent pairs is the H2 test (Phase 6), which uses the matched design and all pairs, not these examples.